In [2]:
# Import Libraries

import pandas as pd
import json
import sqlite3
import os

In [3]:
#Step 1: Load Orders CSV
orders = pd.read_csv("orders.csv")

print("Orders Data")
display(orders.head())
print(orders.columns)


Orders Data


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name'],
      dtype='object')


In [46]:
#Step 2: Load Users JSON
with open("users.json", "r") as f:
    users_data = json.load(f)

users = pd.DataFrame(users_data)

print("Users Data")
display(users.head())
print(users.columns)

Users Data


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


Index(['user_id', 'name', 'city', 'membership'], dtype='object')


In [4]:
conn = sqlite3.connect("restaurants.db")

# Check if table exists
table_check = conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='restaurants';").fetchone()

if table_check is None:
    with open("restaurants.sql", "r") as f:
        sql_script = f.read()
    conn.executescript(sql_script)
else:
    print("Restaurants table already exists, skipping creation.")

# Read the table
restaurants = pd.read_sql_query("SELECT * FROM restaurants;", conn)
print("Restaurants Data Loaded")


Restaurants table already exists, skipping creation.
Restaurants Data Loaded


In [7]:
# Clean column names (strip spaces + lowercase)
orders.columns = orders.columns.str.strip().str.lower()
users.columns = users.columns.str.strip().str.lower()
restaurants.columns = restaurants.columns.str.strip().str.lower()


In [8]:
#Strip extra spaces in string columns
for df, cols in zip([orders, users, restaurants],
                    [['restaurant_name_x'], ['name','membership','city'], ['cuisine','city','restaurant_name_y']]):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()


In [9]:
#Convert dates to datetime
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')


C:\Users\HP\AppData\Local\Temp\ipykernel_6324\2533473298.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')


In [10]:
#Remove duplicate rows
orders = orders.drop_duplicates()
users = users.drop_duplicates(subset='user_id')
restaurants = restaurants.drop_duplicates(subset='restaurant_id')


In [12]:

# Fill missing strings (for merging)
users['membership'] = users['membership'].fillna('Unknown')
users['city'] = users['city'].fillna('Unknown')

if 'cuisine' in restaurants.columns:
    restaurants['cuisine'] = restaurants['cuisine'].fillna('Unknown')
if 'restaurant_name_y' in restaurants.columns:
    restaurants['restaurant_name_y'] = restaurants['restaurant_name_y'].fillna('Unknown')


In [13]:
#Ensure numeric columns are proper type
orders['total_amount'] = pd.to_numeric(orders['total_amount'], errors='coerce')
restaurants['rating'] = pd.to_numeric(restaurants['rating'], errors='coerce')


In [14]:
#Step 5: Merge Orders + Users (LEFT JOIN)

orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)

In [15]:
#Step 6: Merge + Restaurants (LEFT JOIN)

final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)


In [ ]:
#Fix rating_range (include all bins, no empty values)
final_df['rating'] = pd.to_numeric(final_df['rating'], errors='coerce')
final_df['rating'] = final_df['rating'].fillna(0)

bins = [0, 3.5, 4.0, 4.5, 5.0]  # start at 0 to include missing
labels = ['3.0–3.5', '3.6–4.0', '4.1–4.5', '4.6–5.0']

final_df['rating_range'] = pd.cut(
    final_df['rating'],
    bins=bins,
    labels=labels,
    include_lowest=True
)


In [42]:
# Ensure rating is numeric
final_df['rating'] = pd.to_numeric(final_df['rating'], errors='coerce')
final_df['rating'] = final_df['rating'].fillna(0)

# Create rating_range column
bins = [0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0–3.5', '3.6–4.0', '4.1–4.5', '4.6–5.0']

final_df['rating_range'] = pd.cut(
    final_df['rating'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Fill missing values
final_df['rating_range'] = final_df['rating_range'].fillna('3.0–3.5')

# Test
rating_revenue = final_df.groupby('rating_range')['total_amount'].sum()
print(rating_revenue)


rating_range
3.0–3.5    2136772.70
3.6–4.0    1717494.41
4.1–4.5    1960326.26
4.6–5.0    2197030.75
Name: total_amount, dtype: float64


C:\Users\HP\AppData\Local\Temp\ipykernel_6324\1398005770.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rating_revenue = final_df.groupby('rating_range')['total_amount'].sum()


In [17]:
#Add quarter column for order_date
final_df['order_date'] = pd.to_datetime(final_df['order_date'], errors='coerce')
final_df['quarter'] = final_df['order_date'].dt.quarter


In [21]:
#rename
final_df_clean = final_df[[
    'order_id', 
    'order_date', 
    'total_amount', 
    'user_id', 
    'name', 
    'city',            # user city
    'membership', 
    'restaurant_id', 
    'restaurant_name_x', # human-readable restaurant name
    'cuisine', 
    'rating', 
    'restaurant_name_y', # restaurant original name
    'rating_range',
    'quarter'
]].rename(columns={
    'restaurant_name_x': 'restaurant_name'
})

In [23]:
gold_df = final_df[final_df['membership'] == 'Gold']

gold_city_revenue = (
    gold_df.groupby('city')['total_amount']
    .sum()
    .sort_values(ascending=False)
)

gold_city_revenue


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [24]:
#Cuisine with highest average order value
final_df.groupby('cuisine')['total_amount'].mean().sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [25]:
#Users whose total orders > ₹1000
user_total = final_df.groupby('user_id')['total_amount'].sum()
count_users = user_total[user_total > 1000].count()
count_users

2544

In [26]:
#Rating range with highest revenue
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ['3.0–3.5', '3.6–4.0', '4.1–4.5', '4.6–5.0']

final_df['rating_range'] = pd.cut(final_df['rating'], bins=bins, labels=labels)

final_df.groupby('rating_range')['total_amount'].sum().sort_values(ascending=False)


C:\Users\HP\AppData\Local\Temp\ipykernel_6324\2975699642.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  final_df.groupby('rating_range')['total_amount'].sum().sort_values(ascending=False)


rating_range
4.6–5.0    2197030.75
4.1–4.5    1960326.26
3.0–3.5    1881754.57
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [27]:
#Among Gold members, city with highest average order value
gold_df.groupby('city')['total_amount'].mean().sort_values(ascending=False)


city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [28]:
#Cuisine with lowest restaurants but good revenue
rest_count = final_df.groupby('cuisine')['restaurant_id'].nunique()
revenue = final_df.groupby('cuisine')['total_amount'].sum()

pd.DataFrame({
    'restaurant_count': rest_count,
    'revenue': revenue
}).sort_values('restaurant_count')

,restaurant_count,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [29]:
#% of orders by Gold members
gold_orders = len(final_df[final_df['membership'] == 'Gold'])
total_orders = len(final_df)

round((gold_orders / total_orders) * 100)

50

In [30]:
#Restaurant with highest avg order value but < 20 orders
rest_stats = final_df.groupby('restaurant_name_y').agg({
    'total_amount': 'mean',
    'order_id': 'count'
})

rest_stats[rest_stats['order_id'] < 20].sort_values('total_amount', ascending=False)

,total_amount,order_id
restaurant_name_y,,
Restaurant_294,1040.222308,13
Restaurant_262,1029.473333,18
Restaurant_77,1029.180833,12
Restaurant_193,1026.306667,15
Restaurant_7,1002.140625,16
...,...,...
Restaurant_184,621.828947,19
Restaurant_498,596.815556,18
Restaurant_192,589.972857,14


In [31]:
#Restaurant with highest avg order value but < 20 orders
rest_stats = final_df.groupby('restaurant_name_x').agg({
    'total_amount': 'mean',
    'order_id': 'count'
})

rest_stats_filtered = rest_stats[rest_stats['order_id'] < 20] \
                        .sort_values('total_amount', ascending=False)

rest_stats_filtered.head(10)


,total_amount,order_id
restaurant_name_x,,
Hotel Dhaba Multicuisine,1040.222308,13
Sri Mess Punjabi,1029.180833,12
Ruchi Biryani Punjabi,1002.140625,16
Sri Delights Pure Veg,989.467222,18
Classic Kitchen Family Restaurant,973.167895,19
Hotel Dhaba Chinese,973.125556,18
Amma Mess Pure Veg,965.299444,18
Hotel Biryani Pure Veg,964.577692,13
Annapurna Curry House Multicuisine,954.512353,17


In [32]:
#Combination with highest revenue
final_df.groupby(['membership', 'cuisine'])['total_amount'].sum().sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [33]:
#Quarter with highest revenue
final_df['order_date'] = pd.to_datetime(final_df['order_date'])
final_df['quarter'] = final_df['order_date'].dt.quarter

final_df.groupby('quarter')['total_amount'].sum().sort_values(ascending=False)

quarter
3    2037385.10
4    2018263.66
1    2010626.64
2    1945348.72
Name: total_amount, dtype: float64

In [34]:
#Total orders by Gold
len(final_df[final_df['membership'] == 'Gold'])


4987

In [35]:
#Revenue from Hyderabad
round(final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum())


1889367

In [36]:
#Distinct users
final_df['user_id'].nunique()


2883

In [37]:
#Avg order value for Gold
round(final_df[final_df['membership']=='Gold']['total_amount'].mean(), 2)


797.15

In [38]:
#Orders with rating ≥ 4.5
len(final_df[final_df['rating'] >= 4.5])


3374

In [39]:
#Orders in top Gold city
top_city = gold_city_revenue.index[0]
len(gold_df[gold_df['city'] == top_city])


1337

In [40]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)
